In [7]:
# 判斷是否能寬出人
import cv2
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # 小模型先跑得動

cap = cv2.VideoCapture(0)
while True:
    ok, frame = cap.read()
    if not ok:
        break

    results = model(frame, verbose=False)[0]
    for box in results.boxes:
        cls = int(box.cls[0])
        if cls != 0:  # coco class 0 = person
            continue
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(frame, "person", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.imshow("yolo person", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [ ]:
# 人的姿態
import cv2
import mediapipe as mp

mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
with mp_pose.Pose(model_complexity=1, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = pose.process(rgb)

        if res.pose_landmarks:
            mp_draw.draw_landmarks(frame, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)

        cv2.imshow("pose", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

c:\Users\user\anaconda3\envs\pushup\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


KeyboardInterrupt: 

In [11]:
import cv2
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

while True:
    ok, frame = cap.read()
    if not ok:
        break

    r = model.predict(frame, imgsz=320, conf=0.25, classes=[0], max_det=1, verbose=False)[0]

    for box in r.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(frame, "person", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.imshow("yolo person", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [12]:
import cv2
import numpy as np
from ultralytics import YOLO
import mediapipe as mp

yolo = YOLO("yolov8n.pt")
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

with mp_pose.Pose(model_complexity=1, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        h, w = frame.shape[:2]

        det = yolo.predict(frame, imgsz=320, conf=0.25, classes=[0], max_det=1, verbose=False)[0]

        if len(det.boxes) > 0:
            x1, y1, x2, y2 = map(int, det.boxes[0].xyxy[0])
            pad = 20
            x1p = max(0, x1-pad); y1p = max(0, y1-pad)
            x2p = min(w, x2+pad); y2p = min(h, y2+pad)

            roi = frame[y1p:y2p, x1p:x2p]
            rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
            res = pose.process(rgb)

            cv2.rectangle(frame, (x1p,y1p), (x2p,y2p), (0,255,0), 2)

            if res.pose_landmarks:
                # 把 ROI 的 landmark 畫回 ROI 上（比較直覺）
                mp_draw.draw_landmarks(roi, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                frame[y1p:y2p, x1p:x2p] = roi

        cv2.imshow("yolo+pose roi", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [13]:
def angle_3pts(a, b, c):
    import numpy as np
    a = np.array(a); b = np.array(b); c = np.array(c)
    ba = a - b
    bc = c - b
    cosang = np.dot(ba, bc) / (np.linalg.norm(ba)*np.linalg.norm(bc) + 1e-6)
    cosang = np.clip(cosang, -1.0, 1.0)
    return np.degrees(np.arccos(cosang))

In [ ]:
import cv2
import numpy as np
import time
from ultralytics import YOLO
import mediapipe as mp

def angle_3pts(a, b, c):
    a = np.array(a); b = np.array(b); c = np.array(c)
    ba = a - b
    bc = c - b
    cosang = np.dot(ba, bc) / (np.linalg.norm(ba)*np.linalg.norm(bc) + 1e-6)
    cosang = np.clip(cosang, -1.0, 1.0)
    return np.degrees(np.arccos(cosang))

yolo = YOLO("yolov8n.pt")

mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

# thresholds (你之後可調)
UP_TH = 160
DOWN_TH = 100

# smoothing + debounce
alpha = 0.2          # EMA: 越小越平滑
min_interval = 0.4   # 秒，計數後最短間隔

count = 0
state = "UP"
angle_ema = None
last_count_time = 0.0

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

with mp_pose.Pose(model_complexity=1, min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        h, w = frame.shape[:2]

        det = yolo.predict(frame, imgsz=320, conf=0.25, classes=[0], max_det=1, verbose=False)[0]

        elbow_angle = None

        if len(det.boxes) > 0:
            x1, y1, x2, y2 = map(int, det.boxes[0].xyxy[0])
            pad = 20
            x1p = max(0, x1-pad); y1p = max(0, y1-pad)
            x2p = min(w, x2+pad); y2p = min(h, y2+pad)

            roi = frame[y1p:y2p, x1p:x2p]
            rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
            res = pose.process(rgb)

            cv2.rectangle(frame, (x1p,y1p), (x2p,y2p), (0,255,0), 2)

            if res.pose_landmarks:
                lm = res.pose_landmarks.landmark

                # 取 ROI 裡 landmark 轉回原圖座標
                def pt(idx):
                    px = int(lm[idx].x * (x2p-x1p) + x1p)
                    py = int(lm[idx].y * (y2p-y1p) + y1p)
                    return (px, py)

                # 可見度檢查（避免亂跳）
                def vis_ok(idx, th=0.5):
                    return lm[idx].visibility >= th

                # 左: 11肩 13肘 15腕；右: 12肩 14肘 16腕
                needed = [11,13,15, 12,14,16]
                if all(vis_ok(i, 0.5) for i in needed):
                    Ls, Le, Lw = pt(11), pt(13), pt(15)
                    Rs, Re, Rw = pt(12), pt(14), pt(16)

                    left_angle  = angle_3pts(Ls, Le, Lw)
                    right_angle = angle_3pts(Rs, Re, Rw)
                    elbow_angle = (left_angle + right_angle) / 2.0

                    # 畫骨架在 ROI 上
                    mp_draw.draw_landmarks(roi, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                    frame[y1p:y2p, x1p:x2p] = roi

        # 平滑角度
        if elbow_angle is not None:
            if angle_ema is None:
                angle_ema = elbow_angle
            else:
                angle_ema = (1 - alpha) * angle_ema + alpha * elbow_angle

            # 狀態機 + 防抖
            now = time.time()
            if state == "UP" and angle_ema < DOWN_TH:
                state = "DOWN"
            elif state == "DOWN" and angle_ema > UP_TH:
                if now - last_count_time >= min_interval:
                    count += 1
                    last_count_time = now
                state = "UP"

            cv2.putText(frame, f"angle={angle_ema:.1f}", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)

        cv2.putText(frame, f"state={state}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)
        cv2.putText(frame, f"count={count}", (20, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0,255,255), 3)

        cv2.imshow("pushup counter", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [7]:
import cv2
import numpy as np
import time
from ultralytics import YOLO
import mediapipe as mp

# =========================
# 1) 小工具：算兩點距離
# =========================
def dist(a, b):
    """
    計算 2D 點 a, b 的歐式距離
    a, b: (x, y)
    """
    a = np.array(a)
    b = np.array(b)
    return float(np.linalg.norm(a - b))


# =========================
# 2) 載入 YOLO & MediaPipe
# =========================
# YOLOv8 預訓練模型：yolov8n.pt (n = nano，快但相對不如 s/m 準)
# 會用來「只偵測 person」，拿到人框 (bounding box)
yolo = YOLO("yolov8n.pt")

# MediaPipe Pose：用來抓 33 個人體關節點
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils


# =========================
# 3) 深蹲判斷用的門檻（可調）
# =========================
# 我們用特徵：
#   d_norm = distance(hip_mid, knee_mid) / scale
# 站直：hip-knee 距離大 => d_norm 大
# 蹲下：hip-knee 距離小 => d_norm 小
#
# 狀態機：
#   UP -> DOWN：d_norm < DOWN_TH
#   DOWN -> UP：d_norm > UP_TH  (這一刻 count+1)
#
# 先給「偏寬鬆」的初始值：你可依 d_norm 的實際範圍調整
UP_TH = 0.6
DOWN_TH = 0.4


# =========================
# 4) 平滑 & 防抖（可調）
# =========================
alpha = 0.2          # EMA 平滑係數：越小越平滑
min_interval = 0.35  # 兩次計數之間最短間隔（秒）


# =========================
# 5) 狀態機變數
# =========================
count = 0
state = "UP"         # "UP" 站立, "DOWN" 蹲下
d_ema = None         # 平滑後的 d_norm
last_count_time = 0.0


# =========================
# 6) 開相機（OpenCV）
# =========================
cap = cv2.VideoCapture(0)  # 0 = 第一個攝影機
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)   # 固定解析度，讓效能穩定
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)


# =========================
# 7) 建立 MediaPipe Pose 物件
# =========================
with mp_pose.Pose(
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as pose:

    while True:
        # 7.1 讀取一幀畫面
        ok, frame = cap.read()
        if not ok:
            break

        h, w = frame.shape[:2]

        # 每一幀都先清空本幀的 d_norm
        d_norm = None

        # =========================
        # 8) YOLO 偵測「人」
        # =========================
        # yolo.predict(...) 回傳一個 list（因為可以一次推論多張）
        # 我們只給一張 frame，所以取 [0]
        #
        # imgsz=320      : 輸入縮放，越小越快
        # conf=0.25      : 信心分數門檻
        # classes=[0]    : 只偵測 COCO 的 person 類別（class 0）
        # max_det=1      : 只取最多 1 個人（加速 + 你只做單人運動）
        det = yolo.predict(
            frame,
            imgsz=320,
            conf=0.25,
            classes=[0],
            max_det=1,
            verbose=False
        )[0]

        # =========================
        # 9) 若偵測到人，裁切 ROI 後跑 Pose
        # =========================
        if len(det.boxes) > 0:
            # 9.1 取出第一個框的座標 (x1,y1,x2,y2)
            x1, y1, x2, y2 = map(int, det.boxes[0].xyxy[0])

            # 9.2 框外加 padding，避免手腳被截掉
            pad = 20
            x1p = max(0, x1 - pad)
            y1p = max(0, y1 - pad)
            x2p = min(w, x2 + pad)
            y2p = min(h, y2 + pad)

            # 9.3 取 ROI（人框內部）
            roi = frame[y1p:y2p, x1p:x2p]

            # 9.4 MediaPipe 需要 RGB
            rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

            # 9.5 Pose 推論：抓出關節點
            res = pose.process(rgb)

            # 9.6 畫出 ROI 框
            cv2.rectangle(frame, (x1p, y1p), (x2p, y2p), (0, 255, 0), 2)

            # =========================
            # 10) 若 Pose 有抓到關節點
            # =========================
            if res.pose_landmarks:
                lm = res.pose_landmarks.landmark  # 33 個點

                # 10.1 把 ROI 內的相對座標（0~1）轉成原圖像素座標
                def pt(idx):
                    px = int(lm[idx].x * (x2p - x1p) + x1p)
                    py = int(lm[idx].y * (y2p - y1p) + y1p)
                    return (px, py)

                # 10.2 可見度檢查：避免點太飄造成誤判
                def vis_ok(idx, th=0.5):
                    return lm[idx].visibility >= th

                # 10.3 深蹲需要：左右肩(11,12)、左右髖(23,24)、左右膝(25,26)
                needed = [11, 12, 23, 24, 25, 26]

                if all(vis_ok(i, 0.5) for i in needed):
                    # 10.4 把點取出來
                    Ls, Rs = pt(11), pt(12)  # shoulders
                    Lh, Rh = pt(23), pt(24)  # hips
                    Lk, Rk = pt(25), pt(26)  # knees

                    # 10.5 用左右中點減少左右不對稱影響
                    shoulder_mid = ((Ls[0] + Rs[0]) // 2, (Ls[1] + Rs[1]) // 2)
                    hip_mid = ((Lh[0] + Rh[0]) // 2, (Lh[1] + Rh[1]) // 2)
                    knee_mid = ((Lk[0] + Rk[0]) // 2, (Lk[1] + Rk[1]) // 2)

                    # 10.6 用肩到髖的距離作為尺度（抵消遠近）
                    torso_len = dist(shoulder_mid, hip_mid)
                    scale = max(torso_len, 1.0)  # 防止除以 0

                    # 10.7 深蹲特徵：hip-knee 距離（蹲下變小，站起變大）
                    d = dist(hip_mid, knee_mid)
                    d_norm = d / scale

                    # 10.8 畫骨架（畫在 ROI 上，再貼回原圖）
                    mp_draw.draw_landmarks(roi, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                    frame[y1p:y2p, x1p:x2p] = roi

                    # 10.9 畫 hip/knee 中點與連線，方便觀察
                    cv2.circle(frame, hip_mid, 7, (0, 255, 255), -1)    # 黃：髖中點
                    cv2.circle(frame, knee_mid, 7, (255, 255, 255), -1) # 白：膝中點
                    cv2.line(frame, hip_mid, knee_mid, (0, 255, 255), 2)

        # =========================
        # 11) d_norm 平滑（EMA）+ 狀態機計數
        # =========================
        if d_norm is not None:
            # 11.1 EMA 平滑
            if d_ema is None:
                d_ema = d_norm
            else:
                d_ema = (1 - alpha) * d_ema + alpha * d_norm

            # 11.2 狀態機 + 防抖
            now = time.time()

            # 站立(UP) -> 蹲下(DOWN)：d_ema 變小到小於 DOWN_TH
            if state == "UP" and d_ema < DOWN_TH:
                state = "DOWN"

            # 蹲下(DOWN) -> 站立(UP)：d_ema 變大到大於 UP_TH
            elif state == "DOWN" and d_ema > UP_TH:
                # 防抖：避免短時間內重複計數
                if now - last_count_time >= min_interval:
                    count += 1
                    last_count_time = now
                state = "UP"

            # 顯示目前 d_norm
            cv2.putText(frame, f"d_norm={d_ema:.2f}", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
        else:
            cv2.putText(frame, "d_norm=--", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

        # =========================
        # 12) 顯示 UI：狀態 & 次數 & 門檻
        # =========================
        cv2.putText(frame, f"state={state}", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

        cv2.putText(frame, f"count={count}", (20, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        # cv2.putText(frame, f"UP if d>{UP_TH:.2f}, DOWN if d<{DOWN_TH:.2f}", (20, 180),
        #             cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

        # 12.1 顯示畫面
        cv2.imshow("squat counter (YOLO+Pose)", frame)

        # 12.2 按 ESC 離開
        if cv2.waitKey(1) & 0xFF == 27:
            break

# =========================
# 13) 收尾：釋放資源
# =========================
cap.release()
cv2.destroyAllWindows()